# Download FineWeb-Edu Raw Data

Downloads ~2B tokens worth of raw text from FineWeb-Edu.
Saves raw text so you can tokenize with different tokenizers later.

**Output:** Raw text dataset with train/val/test splits

In [ ]:
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import random

# Configuration
OUTPUT_DIR = Path("../data/fineweb-edu-raw")
TARGET_TOKENS = 2_000_000_000  # ~2B tokens (estimated)
SEED = 42

# We use a tokenizer just to estimate token count, not to save tokens
ESTIMATOR_TOKENIZER = "gpt2"

## 1. Stream and Download Raw Text

In [ ]:
# Load tokenizer just for estimating token counts
tokenizer = AutoTokenizer.from_pretrained(ESTIMATOR_TOKENIZER)
print(f"Using {ESTIMATOR_TOKENIZER} to estimate token counts")

In [ ]:
# Stream FineWeb-Edu
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    split="train",
    streaming=True,
)

print("Streaming dataset loaded")

In [ ]:
# Collect raw text documents until we reach target tokens
documents = []
total_tokens = 0

pbar = tqdm(total=TARGET_TOKENS, unit="tok", desc="Collecting data")

for i, example in enumerate(dataset):
    text = example["text"]
    
    # Estimate token count (for progress tracking)
    est_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    
    # Skip very short documents
    if est_tokens < 50:
        continue
    
    # Save RAW TEXT, not tokens
    documents.append({
        "text": text,
        "uid": len(documents),
    })
    
    total_tokens += est_tokens
    pbar.update(est_tokens)
    
    if total_tokens >= TARGET_TOKENS:
        break
    
    if len(documents) % 10000 == 0:
        pbar.set_postfix({"docs": len(documents)})

pbar.close()

print(f"\nCollected {len(documents):,} documents")
print(f"Estimated tokens: {total_tokens:,} ({total_tokens/1e9:.2f}B)")

## 2. Create Train/Val/Test Splits

In [ ]:
# Shuffle
random.seed(SEED)
random.shuffle(documents)

# Split: 95% train, 2.5% val, 2.5% test
n = len(documents)
train_end = int(0.95 * n)
val_end = int(0.975 * n)

train_docs = documents[:train_end]
val_docs = documents[train_end:val_end]
test_docs = documents[val_end:]

print(f"Train: {len(train_docs):,} documents")
print(f"Val:   {len(val_docs):,} documents")
print(f"Test:  {len(test_docs):,} documents")

## 3. Save Raw Text Data

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset = Dataset.from_list(train_docs)
val_dataset = Dataset.from_list(val_docs)
test_dataset = Dataset.from_list(test_docs)

train_dataset.save_to_disk(OUTPUT_DIR / "train")
val_dataset.save_to_disk(OUTPUT_DIR / "val")
test_dataset.save_to_disk(OUTPUT_DIR / "test")

print(f"Saved raw text to {OUTPUT_DIR}")

## 4. Verify

In [ ]:
from datasets import load_from_disk

check = load_from_disk(str(OUTPUT_DIR / "train"))
print(f"Columns: {check.column_names}")
print(f"\nFirst document preview:")
print(check[0]["text"][:500] + "...")

## Done!

Raw text saved to `data/fineweb-edu-raw/`

Now use notebook `02_tokenize_data.ipynb` to tokenize with any tokenizer.